<a href="https://colab.research.google.com/github/StathisDevves/Industrial/blob/main/%CE%91%CE%BD%CF%84%CE%AF%CE%B3%CF%81%CE%B1%CF%86%CE%BF_Untitled1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
from openpyxl import load_workbook
from openpyxl.styles import Font, PatternFill, Alignment, Border, Side

# =========================================================
# INDUSTRIAL MODEL MINI MILL - COST MINIMIZING SCHEDULE
# =========================================================
# INPUT:
#   Excel file with columns:
#       1st column = Hour
#       2nd column = Price
#
# OUTPUT:
#   Excel workbook with:
#       - Summary
#       - Input Prices
#       - Stage Sequence
#       - Optimization
#       - Optimized Schedule
#       - Notes
#
# LOGIC:
#   We do NOT simply pick the cheapest 4-hour EAF block.
#   Instead, we optimize the START HOUR of Step 1 so that
#   the TOTAL COST = sum(Load_stage * Price_hour) is minimized
#   across the full 24-hour circular sequence.
# =========================================================

# -----------------------------
# 1. FILE PATHS
# -----------------------------
input_file = "Production sequense to Prices 19March 2025 set.xlsx"
output_file = "Industrial_Model_Mini_Mill_Optimized_Schedule.xlsx"

# -----------------------------
# 2. READ INPUT
# -----------------------------
df_input = pd.read_excel(input_file)

# Keep first two columns only
df_input = df_input.iloc[:, :2].copy()
df_input.columns = ["Hour", "Price"]

# Basic checks
if len(df_input) != 24:
    raise ValueError("The input file must contain exactly 24 hourly rows.")

# Sort by hour if needed
df_input = df_input.sort_values("Hour").reset_index(drop=True)

hours = df_input["Hour"].tolist()
prices = df_input["Price"].tolist()
n = len(df_input)

# -----------------------------
# 3. DEFINE PROCESS SEQUENCE
# -----------------------------
# Full circular 24-hour sequence:
# Step 1: 4 hrs  -> EAF1.1 ... EAF1.4
# Step 2: 3 hrs  -> Secondary Downstream 1.1 ... 1.3
# Step 3: 1 hr   -> SD sequence Blue Colour
# Step 4: 6 hrs  -> Minimum Critical Load 1 ... 6
# Step 5: 3 hrs  -> Preparation Blue 1 ... 3
# Step 6: 4 hrs  -> EAF2.1 ... EAF2.4
# Step 7: 3 hrs  -> Secondary Stream 2.1 ... 2.3

stage_sequence = [
    ("Step 1", "EAF1.1", 72),
    ("Step 1", "EAF1.2", 78),
    ("Step 1", "EAF1.3", 80),
    ("Step 1", "EAF1.4", 72),

    ("Step 2", "Secondary Downstream 1.1", 32),
    ("Step 2", "Secondary Downstream 1.2", 26),
    ("Step 2", "Secondary Downstream 1.3", 20),

    ("Step 3", "SD sequence Blue Colour", 15),

    ("Step 4", "Minimum Critical Load 1", 12),
    ("Step 4", "Minimum Critical Load 2", 12),
    ("Step 4", "Minimum Critical Load 3", 12),
    ("Step 4", "Minimum Critical Load 4", 12),
    ("Step 4", "Minimum Critical Load 5", 12),
    ("Step 4", "Minimum Critical Load 6", 12),

    ("Step 5", "Preparation Blue 1", 15),
    ("Step 5", "Preparation Blue 2", 24),
    ("Step 5", "Preparation Blue 3", 38),

    ("Step 6", "EAF2.1", 72),
    ("Step 6", "EAF2.2", 78),
    ("Step 6", "EAF2.3", 80),
    ("Step 6", "EAF2.4", 76),

    ("Step 7", "Secondary Stream 2.1", 28),
    ("Step 7", "Secondary Stream 2.2", 24),
    ("Step 7", "Secondary Stream 2.3", 20),
]

if len(stage_sequence) != 24:
    raise ValueError("Stage sequence must contain exactly 24 rows.")

df_sequence = pd.DataFrame(stage_sequence, columns=["Step", "Production Phase", "Total Load"])

# -----------------------------
# 4. COST FUNCTION
# -----------------------------
def build_schedule_for_start(start_hour_index, prices_24, hours_24, sequence):
    """
    Maps the 24-stage production sequence to the 24 hourly prices,
    starting from start_hour_index in circular fashion.
    """
    rows = []
    total_cost = 0.0

    for i, (step, phase, load) in enumerate(sequence):
        idx = (start_hour_index + i) % 24
        hour = hours_24[idx]
        price = prices_24[idx]
        cost = load * price
        total_cost += cost

        rows.append({
            "Sequence Position": i + 1,
            "Step": step,
            "Hour": hour,
            "Price": price,
            "Production Phase": phase,
            "Total Load (MWh)": load,
            "Cost = Load x Price": cost
        })

    return pd.DataFrame(rows), total_cost

# -----------------------------
# 5. OPTIMIZE OVER ALL 24 START HOURS
# -----------------------------
optimization_rows = []
best_start_index = None
best_cost = None
best_schedule = None

for start_idx in range(24):
    schedule_df, total_cost = build_schedule_for_start(start_idx, prices, hours, stage_sequence)

    optimization_rows.append({
        "Candidate Start Hour for EAF1.1": hours[start_idx],
        "Total Cost": total_cost
    })

    if best_cost is None or total_cost < best_cost:
        best_cost = total_cost
        best_start_index = start_idx
        best_schedule = schedule_df.copy()

df_optimization = pd.DataFrame(optimization_rows).sort_values("Candidate Start Hour for EAF1.1").reset_index(drop=True)

# -----------------------------
# 6. SUMMARY
# -----------------------------
best_start_hour = hours[best_start_index]

df_summary = pd.DataFrame({
    "Metric": [
        "Optimal Step 1 start hour (EAF1.1)",
        "Minimum total daily cost",
        "Optimization criterion",
        "Number of candidate start hours tested",
    ],
    "Value": [
        best_start_hour,
        best_cost,
        "Minimize sum of (Load per Process Stage × Electricity Price at assigned hour)",
        24,
    ]
})

# -----------------------------
# 7. SAVE TO EXCEL
# -----------------------------
with pd.ExcelWriter(output_file, engine="openpyxl") as writer:
    df_summary.to_excel(writer, sheet_name="Summary", index=False)
    df_input.to_excel(writer, sheet_name="Input Prices", index=False)
    df_sequence.to_excel(writer, sheet_name="Stage Sequence", index=False)
    df_optimization.to_excel(writer, sheet_name="Optimization", index=False)
    best_schedule.to_excel(writer, sheet_name="Optimized Schedule", index=False)

    notes_df = pd.DataFrame({
        "Notes": [
            "The model tests all 24 possible start hours for Step 1 (EAF1.1).",
            "The production sequence is fixed and continues circularly over the 24-hour day.",
            "For each candidate start hour, every process stage is mapped to one hour and one electricity price.",
            "The objective is to minimize total cost = sum(Total Load × Price).",
            "The optimized schedule sheet shows the least-cost mapping."
        ]
    })
    notes_df.to_excel(writer, sheet_name="Notes", index=False)

# -----------------------------
# 8. FORMAT WORKBOOK
# -----------------------------
wb = load_workbook(output_file)

header_fill = PatternFill("solid", fgColor="1F4E78")
header_font = Font(color="FFFFFF", bold=True)
thin = Side(style="thin", color="BFBFBF")
border = Border(left=thin, right=thin, top=thin, bottom=thin)

phase_fills = {
    "EAF": PatternFill("solid", fgColor="FCE4D6"),
    "Secondary": PatternFill("solid", fgColor="E2F0D9"),
    "SD sequence Blue Colour": PatternFill("solid", fgColor="D9EAF7"),
    "Minimum Critical Load": PatternFill("solid", fgColor="F4CCCC"),
    "Preparation Blue": PatternFill("solid", fgColor="D9E1F2"),
}

for ws in wb.worksheets:
    # Header formatting
    for cell in ws[1]:
        cell.fill = header_fill
        cell.font = header_font
        cell.alignment = Alignment(horizontal="center", vertical="center")
        cell.border = border

    # Body formatting
    for row in ws.iter_rows(min_row=2):
        for cell in row:
            cell.border = border
            if isinstance(cell.value, (int, float)):
                if "Cost" in str(ws.cell(1, cell.column).value) or "Price" in str(ws.cell(1, cell.column).value):
                    cell.number_format = "0.00"
            cell.alignment = Alignment(vertical="center")

    # Auto width
    for col_cells in ws.columns:
        max_len = 0
        col_letter = col_cells[0].column_letter
        for cell in col_cells:
            val = "" if cell.value is None else str(cell.value)
            if len(val) > max_len:
                max_len = len(val)
        ws.column_dimensions[col_letter].width = min(max_len + 2, 35)

# Color the optimized schedule phases
ws_opt = wb["Optimized Schedule"]
headers = [c.value for c in ws_opt[1]]
phase_col = headers.index("Production Phase") + 1

for row in range(2, ws_opt.max_row + 1):
    phase_value = str(ws_opt.cell(row=row, column=phase_col).value)

    fill_to_use = None
    if phase_value.startswith("EAF"):
        fill_to_use = phase_fills["EAF"]
    elif phase_value.startswith("Secondary"):
        fill_to_use = phase_fills["Secondary"]
    elif phase_value == "SD sequence Blue Colour":
        fill_to_use = phase_fills["SD sequence Blue Colour"]
    elif phase_value.startswith("Minimum Critical Load"):
        fill_to_use = phase_fills["Minimum Critical Load"]
    elif phase_value.startswith("Preparation Blue"):
        fill_to_use = phase_fills["Preparation Blue"]

    if fill_to_use:
        ws_opt.cell(row=row, column=phase_col).fill = fill_to_use

wb.save(output_file)

# -----------------------------
# 9. PRINT RESULTS
# -----------------------------
print(f"Output saved to: {output_file}")
print(f"Optimal Step 1 start hour (EAF1.1): {best_start_hour}")
print(f"Minimum total daily cost: {best_cost:,.2f}")

Output saved to: Industrial_Model_Mini_Mill_Optimized_Schedule.xlsx
Optimal Step 1 start hour (EAF1.1): 8
Minimum total daily cost: 71,674.49
